# Layout Clustering Pipeline vs Standalone Dripper — Comparison

**Dataset**: chunk_0 from host_bucket=0000 — 44K pages, 1,424 layout IDs  
**Run A**: Dripper with layout clustering (template propagation)  
**Run B**: Standalone Dripper (LLM on every page, no clustering)  

### Sections
0. Setup & Configuration  
1. Load Results  
2. LLM Call Efficiency  
3. Throughput & Cost  
4. Quality — F1 vs Standalone  
5. Per-Host Analysis  
6. Cluster Size Distribution  
7. Example Content Comparison  
8. Summary Scorecard

## 0. Setup & Configuration

In [ ]:
%matplotlib inline
import sys, os, re, json, time, warnings
from pathlib import Path
from collections import Counter

# ── Configurable paths ────────────────────────────────────────────────────────
CURATOR_REPO = "/raid/vjawa/nemo-curator-adlr-mm/submodules/Curator"
DATA_DIR     = "/raid/vjawa/dripper_tutorial"

# Manifest produced by the layout precompute job (chunk_0 / host_bucket=0000)
MANIFEST_PATH = f"{DATA_DIR}/layout_precompute_manifest.parquet"

# ── Run output paths (update these once jobs complete) ────────────────────────
# Run A: Dripper WITH layout clustering
RUN_A_DIR = "/lustre/fsw/portfolios/llmservice/users/vjawa/dripper_cc_main_2025_26_smoke/RUN_A_JOB_ID"

# Run B: Standalone Dripper (no clustering)
RUN_B_DIR = "/lustre/fsw/portfolios/llmservice/users/vjawa/dripper_cc_main_2025_26_smoke/RUN_B_JOB_ID"

RUN_A_RESULTS = f"{RUN_A_DIR}/dripper_results.parquet"
RUN_B_RESULTS = f"{RUN_B_DIR}/dripper_results.parquet"
RUN_A_METRICS = f"{RUN_A_DIR}/metrics.json"
RUN_B_METRICS = f"{RUN_B_DIR}/metrics.json"

# ── Python path ───────────────────────────────────────────────────────────────
sys.path.insert(0, CURATOR_REPO)

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

pd.set_option("display.max_colwidth", 90)
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Helpers ───────────────────────────────────────────────────────────────────
def read_parquet(path):
    """Use ParquetFile directly — avoids ParquetDataset buffer error on pyarrow 23."""
    return pq.ParquetFile(str(path)).read().to_pandas()

def coerce_html(raw):
    if isinstance(raw, bytes):
        return raw.decode("utf-8", errors="replace")
    return str(raw or "")

def load_json_safe(path):
    """Load a JSON file; return empty dict if missing."""
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return {}
    except Exception as e:
        print(f"  Warning: could not read {path}: {e}")
        return {}

def load_parquet_safe(path, label):
    """Load a parquet file with a graceful error if not yet available."""
    try:
        df = read_parquet(path)
        print(f"  {label}: {len(df):,} rows, {len(df.columns)} cols")
        return df
    except FileNotFoundError:
        print(f"  {label}: NOT FOUND — {path}")
        print(f"    (update the path at the top of this notebook once the job completes)")
        return None
    except Exception as e:
        print(f"  {label}: ERROR reading {path}: {e}")
        return None

print("Setup OK")

## 1. Load Results

In [ ]:
print("Loading Run A (with clustering)...")
run_a = load_parquet_safe(RUN_A_RESULTS, "Run A")
metrics_a = load_json_safe(RUN_A_METRICS)
if metrics_a:
    print(f"  metrics_a keys: {list(metrics_a.keys())}")
else:
    print(f"  metrics.json not found at {RUN_A_METRICS}")

print()
print("Loading Run B (standalone)...")
run_b = load_parquet_safe(RUN_B_RESULTS, "Run B")
metrics_b = load_json_safe(RUN_B_METRICS)
if metrics_b:
    print(f"  metrics_b keys: {list(metrics_b.keys())}")
else:
    print(f"  metrics.json not found at {RUN_B_METRICS}")

print()
print("Loading cluster manifest...")
manifest = load_parquet_safe(MANIFEST_PATH, "Manifest")
if manifest is not None:
    print(f"  hosts:      {manifest['url_host_name'].nunique():,}")
    layout_ids = manifest['dripper_layout_id'].dropna()
    n_clustered = layout_ids.str.startswith('layout-', na=False).sum()
    print(f"  layout IDs: {layout_ids.nunique():,}  ({n_clustered:,} clustered rows)")

In [ ]:
# Print schemas and verify URL alignment
if run_a is not None:
    print("Run A columns:", list(run_a.columns))
if run_b is not None:
    print("Run B columns:", list(run_b.columns))
if manifest is not None:
    print("Manifest columns:", list(manifest.columns))

print()
if run_a is not None and run_b is not None:
    overlap = set(run_a['url']) & set(run_b['url'])
    print(f"URL overlap Run A ∩ Run B: {len(overlap):,} pages")
    print(f"  Run A only: {len(set(run_a['url']) - set(run_b['url'])):,}")
    print(f"  Run B only: {len(set(run_b['url']) - set(run_a['url'])):,}")

if run_a is not None and manifest is not None:
    overlap_am = set(run_a['url']) & set(manifest['url'])
    print(f"URL overlap Run A ∩ Manifest: {len(overlap_am):,} pages")

## 2. LLM Call Efficiency

Layout clustering avoids an LLM call for every page in a cluster except the representative.  
The `metrics.json` file records:
- `llm_request_pages` — pages that triggered an actual LLM call
- `layout_template_saved_call_pages` — pages whose results came from template propagation
- `total_tokens` — total prompt + completion tokens consumed

In [ ]:
def get_metric(m, *keys, default=0):
    """Retrieve a metric by one of several possible key names."""
    for k in keys:
        if k in m:
            return m[k]
    return default

# Pull metrics (fall back to run_a/run_b row counts when metrics.json is missing)
total_pages_a = get_metric(metrics_a, 'total_pages',
                           default=len(run_a) if run_a is not None else 0)
total_pages_b = get_metric(metrics_b, 'total_pages',
                           default=len(run_b) if run_b is not None else 0)

llm_calls_a   = get_metric(metrics_a, 'llm_request_pages')
llm_calls_b   = get_metric(metrics_b, 'llm_request_pages',
                            default=total_pages_b)  # standalone = all pages

saved_a       = get_metric(metrics_a, 'layout_template_saved_call_pages')
tokens_a      = get_metric(metrics_a, 'total_tokens')
tokens_b      = get_metric(metrics_b, 'total_tokens')

call_reduction = (1 - llm_calls_a / llm_calls_b) * 100 if llm_calls_b > 0 else 0
token_reduction = (1 - tokens_a / tokens_b) * 100 if tokens_b > 0 else 0

print("LLM Call Summary")
print(f"{'':40s}  {'Run A (clustering)':>20s}  {'Run B (standalone)':>20s}")
print("-" * 85)
print(f"{'Total pages':40s}  {total_pages_a:>20,}  {total_pages_b:>20,}")
print(f"{'LLM calls':40s}  {llm_calls_a:>20,}  {llm_calls_b:>20,}")
print(f"{'Pages saved by template propagation':40s}  {saved_a:>20,}  {'N/A':>20s}")
print(f"{'Total tokens':40s}  {tokens_a:>20,}  {tokens_b:>20,}")
print(f"{'Call reduction vs standalone':40s}  {call_reduction:>19.1f}%  {'baseline':>20s}")
print(f"{'Token reduction vs standalone':40s}  {token_reduction:>19.1f}%  {'baseline':>20s}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

runs  = ["Run A\n(clustering)", "Run B\n(standalone)"]
calls = [llm_calls_a, llm_calls_b]
toks  = [tokens_a,  tokens_b]
pgs   = [total_pages_a, total_pages_b]
colors = ["#5cb85c", "#d9534f"]

# Panel 1: total pages vs LLM calls
ax = axes[0]
x = np.arange(2)
w = 0.35
b1 = ax.bar(x - w/2, pgs,   width=w, label="Total pages",  color="steelblue",  alpha=0.85)
b2 = ax.bar(x + w/2, calls, width=w, label="LLM calls",    color="#f0ad4e", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(runs)
ax.set_title("Pages vs LLM Calls")
ax.set_ylabel("Count")
ax.legend(fontsize=8)
for b in list(b1) + list(b2):
    h = b.get_height()
    if h > 0:
        ax.text(b.get_x() + b.get_width()/2, h * 1.01, f"{h:,.0f}",
                ha="center", va="bottom", fontsize=7)

# Panel 2: call reduction
ax = axes[1]
ax.bar(runs, calls, color=colors, edgecolor="black", linewidth=0.5)
ax.set_title("LLM Calls")
ax.set_ylabel("LLM calls")
for i, (r, c) in enumerate(zip(runs, calls)):
    ax.text(i, c * 1.01, f"{c:,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
if call_reduction > 0:
    ax.set_title(f"LLM Calls  ({call_reduction:.1f}% reduction)")

# Panel 3: tokens
ax = axes[2]
ax.bar(runs, toks, color=colors, edgecolor="black", linewidth=0.5)
ax.set_title("Total Tokens")
ax.set_ylabel("Tokens")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K"))
for i, (r, t) in enumerate(zip(runs, toks)):
    label = f"{t/1e6:.1f}M" if t >= 1e6 else f"{t/1e3:.0f}K"
    ax.text(i, t * 1.01, label, ha="center", va="bottom", fontsize=9, fontweight="bold")
if token_reduction > 0:
    ax.set_title(f"Total Tokens  ({token_reduction:.1f}% reduction)")

fig.suptitle("LLM Call Efficiency — Clustering vs Standalone", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 3. Throughput & Cost

In [ ]:
# Pull timing from metrics.json
elapsed_a = get_metric(metrics_a, 'elapsed_s', 'elapsed_seconds')
elapsed_b = get_metric(metrics_b, 'elapsed_s', 'elapsed_seconds')

throughput_a = total_pages_a / elapsed_a if elapsed_a > 0 else 0
throughput_b = total_pages_b / elapsed_b if elapsed_b > 0 else 0

# H100-hour projection to full CC snapshot (~2.4B pages)
FULL_SNAPSHOT_PAGES = 2_400_000_000
# pages/s → seconds for full snapshot → /3600 for hours
h100h_a = (FULL_SNAPSHOT_PAGES / throughput_a / 3600) if throughput_a > 0 else 0
h100h_b = (FULL_SNAPSHOT_PAGES / throughput_b / 3600) if throughput_b > 0 else 0

rows = [
    {"Metric": "Elapsed (s)",       "Run A (clustering)": f"{elapsed_a:,.0f}",   "Run B (standalone)": f"{elapsed_b:,.0f}"},
    {"Metric": "Throughput (pages/s)","Run A (clustering)": f"{throughput_a:.1f}", "Run B (standalone)": f"{throughput_b:.1f}"},
    {"Metric": "H100-hours (full snapshot)",
     "Run A (clustering)": f"{h100h_a:,.0f}" if h100h_a > 0 else "N/A",
     "Run B (standalone)": f"{h100h_b:,.0f}" if h100h_b > 0 else "N/A"},
]
summary_df = pd.DataFrame(rows).set_index("Metric")
display(summary_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ["#5cb85c", "#d9534f"]
runs   = ["Run A\n(clustering)", "Run B\n(standalone)"]

# Panel 1: throughput
ax = axes[0]
tput = [throughput_a, throughput_b]
bars = ax.bar(runs, tput, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("pages / second")
ax.set_title("Throughput")
for bar, v in zip(bars, tput):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, v * 1.01,
                f"{v:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

# Panel 2: H100-hours
ax = axes[1]
h100s = [h100h_a, h100h_b]
bars = ax.bar(runs, h100s, color=colors, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Projected H100-hours")
ax.set_title("Projected Cost (full CC snapshot, 2.4B pages)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
for bar, v in zip(bars, h100s):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, v * 1.01,
                f"{v/1000:.0f}K", ha="center", va="bottom", fontsize=10, fontweight="bold")

fig.suptitle("Throughput & Projected Cost", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

if h100h_a > 0 and h100h_b > 0:
    cost_reduction = (1 - h100h_a / h100h_b) * 100
    print(f"Cost reduction: {cost_reduction:.1f}%  ({h100h_b - h100h_a:,.0f} H100-hours saved)")

## 4. Quality — F1 vs Standalone

For propagated rows in Run A, we compare the template-propagated content against  
Run B's LLM-extracted content (treated as ground truth) using token bag-of-words F1.

F1 = harmonic mean of token-level precision and recall.  
Target: mean F1 ≥ 0.95.

In [ ]:
try:
    from nemo_curator.stages.text.experimental.dripper.stage import _token_f1
    print("_token_f1 imported OK")
except ImportError as e:
    print(f"Import failed: {e}")
    print("Using local fallback implementation.")
    import re as _re
    def _token_f1(pred: str, ref: str) -> float:
        """Token bag-of-words F1."""
        if not pred and not ref:
            return 1.0
        if not pred or not ref:
            return 0.0
        pred_toks = Counter(_re.findall(r'\w+', pred.lower()))
        ref_toks  = Counter(_re.findall(r'\w+', ref.lower()))
        common = sum((pred_toks & ref_toks).values())
        prec   = common / sum(pred_toks.values())
        rec    = common / sum(ref_toks.values())
        if prec + rec == 0:
            return 0.0
        return 2 * prec * rec / (prec + rec)

In [ ]:
f1_df = None

if run_a is None or run_b is None:
    print("Run A and/or Run B not loaded — skipping F1 analysis.")
    print("Update RUN_A_DIR / RUN_B_DIR at the top of the notebook and re-run.")
else:
    # Identify propagated rows in Run A (not an actual LLM call)
    # Expected column: 'is_propagated' or derive from 'llm_called' flag
    if 'is_propagated' in run_a.columns:
        propagated_a = run_a[run_a['is_propagated'] == True].copy()
    elif 'llm_called' in run_a.columns:
        propagated_a = run_a[run_a['llm_called'] == False].copy()
    else:
        # Fall back: all rows that have a layout_id (template was applied)
        if 'dripper_layout_id' in run_a.columns:
            propagated_a = run_a[run_a['dripper_layout_id'].notna()].copy()
        else:
            propagated_a = run_a.copy()
        print(f"Note: 'is_propagated' / 'llm_called' column not found; "
              f"using all {len(propagated_a):,} rows for F1 analysis.")

    print(f"Propagated rows in Run A: {len(propagated_a):,}")

    # Merge with Run B on URL to get ground-truth content
    content_col_a = next((c for c in ['dripper_content', 'content', 'main_content'] if c in run_a.columns), None)
    content_col_b = next((c for c in ['dripper_content', 'content', 'main_content'] if c in run_b.columns), None)

    if content_col_a is None or content_col_b is None:
        print(f"Content columns not found.")
        print(f"  Run A columns: {list(run_a.columns)}")
        print(f"  Run B columns: {list(run_b.columns)}")
    else:
        print(f"Using '{content_col_a}' from Run A and '{content_col_b}' from Run B")

        merged = propagated_a[['url', content_col_a]].merge(
            run_b[['url', content_col_b]].rename(columns={content_col_b: 'content_b'}),
            on='url', how='inner'
        ).rename(columns={content_col_a: 'content_a'})

        print(f"Merged (propagated A ∩ B): {len(merged):,} rows")

        # Compute F1
        merged['f1'] = merged.apply(
            lambda r: _token_f1(str(r['content_a'] or ''), str(r['content_b'] or '')), axis=1
        )

        # Add host column from manifest if available
        if manifest is not None and 'url_host_name' in manifest.columns:
            merged = merged.merge(manifest[['url', 'url_host_name', 'dripper_layout_id']],
                                  on='url', how='left')

        f1_df = merged
        print(f"\nF1 summary:")
        print(f"  Mean F1:      {f1_df['f1'].mean():.4f}")
        print(f"  Median F1:    {f1_df['f1'].median():.4f}")
        print(f"  Min F1:       {f1_df['f1'].min():.4f}")
        print(f"  F1 >= 0.95:   {(f1_df['f1'] >= 0.95).sum():,} / {len(f1_df):,} "
              f"({(f1_df['f1'] >= 0.95).mean()*100:.1f}%)")
        print(f"  F1 >= 0.90:   {(f1_df['f1'] >= 0.90).sum():,} / {len(f1_df):,} "
              f"({(f1_df['f1'] >= 0.90).mean()*100:.1f}%)")

In [ ]:
if f1_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Full distribution
    ax = axes[0]
    ax.hist(f1_df['f1'], bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(f1_df['f1'].mean(), color='orange', linewidth=2, linestyle='--',
               label=f"Mean: {f1_df['f1'].mean():.3f}")
    ax.axvline(0.95, color='red', linewidth=1.5, linestyle=':',
               label='Threshold: 0.95')
    ax.set_xlabel("Token F1")
    ax.set_ylabel("# propagated pages")
    ax.set_title("F1 Distribution — All Propagated Rows")
    ax.legend()

    # Zoom on low tail (F1 < 0.8)
    ax = axes[1]
    low_f1 = f1_df[f1_df['f1'] < 0.8]
    if len(low_f1) > 0:
        ax.hist(low_f1['f1'], bins=30, color='#d9534f', edgecolor='white', linewidth=0.3)
        ax.set_xlabel("Token F1")
        ax.set_ylabel("# pages")
        ax.set_title(f"Low-F1 Tail (F1 < 0.80) — {len(low_f1):,} pages")
    else:
        ax.text(0.5, 0.5, "No pages with F1 < 0.80", ha='center', va='center',
                fontsize=13, transform=ax.transAxes)
        ax.set_title("Low-F1 Tail (F1 < 0.80)")

    plt.suptitle("Propagation Quality vs Standalone (Run B = ground truth)", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

    # Worst examples
    print("\nWorst 10 propagated examples by F1:")
    worst_cols = ['url', 'f1']
    if 'url_host_name' in f1_df.columns:
        worst_cols = ['url', 'url_host_name', 'f1']
    display(f1_df.nsmallest(10, 'f1')[worst_cols])

## 5. Per-Host Analysis

Which hosts benefited most from clustering?  
Which hosts had the worst propagation quality?

In [ ]:
if manifest is not None:
    # Pages saved = clustered pages minus one representative per cluster
    named = manifest[manifest['dripper_layout_id'].str.startswith('layout-', na=False)].copy()
    cluster_sizes = named.groupby('dripper_layout_id').size().rename('cluster_size')
    named = named.merge(cluster_sizes, on='dripper_layout_id', how='left')

    # Saved calls per cluster = cluster_size - 1 (1 call for representative)
    named['saved_calls'] = named['cluster_size'] - 1

    # Aggregate per host
    host_stats = named.groupby('url_host_name').agg(
        total_pages   = ('url', 'count'),
        n_clusters    = ('dripper_layout_id', 'nunique'),
        saved_calls   = ('saved_calls', 'sum'),
    ).reset_index()
    host_stats['save_rate'] = host_stats['saved_calls'] / host_stats['total_pages']
    host_stats = host_stats.sort_values('saved_calls', ascending=False)

    print(f"Top 15 hosts by saved LLM calls:")
    display(host_stats.head(15).reset_index(drop=True))
else:
    print("Manifest not loaded — skipping per-host saved-calls analysis.")
    host_stats = None

In [ ]:
if f1_df is not None and 'url_host_name' in f1_df.columns:
    host_f1 = f1_df.groupby('url_host_name').agg(
        n_pages  = ('f1', 'count'),
        mean_f1  = ('f1', 'mean'),
        min_f1   = ('f1', 'min'),
        pct_above_95 = ('f1', lambda x: (x >= 0.95).mean() * 100),
    ).reset_index().sort_values('mean_f1')

    print("Hosts with worst mean F1 (bottom 15):")
    display(host_f1.head(15).reset_index(drop=True))

In [ ]:
if host_stats is not None:
    top5_hosts = host_stats.head(5)['url_host_name'].tolist()
    print("Top 5 hosts by saved calls — cluster count, pages, F1 distribution")
    print()

    fig, axes = plt.subplots(1, len(top5_hosts), figsize=(3.5 * len(top5_hosts), 4), sharey=False)
    if len(top5_hosts) == 1:
        axes = [axes]

    for ax, host in zip(axes, top5_hosts):
        host_row = host_stats[host_stats['url_host_name'] == host].iloc[0]
        label = f"{host[:30]}\n{host_row['total_pages']:,} pages\n"\
                f"{host_row['n_clusters']} clusters\n{host_row['saved_calls']:,} saved"

        if f1_df is not None and 'url_host_name' in f1_df.columns:
            hf1 = f1_df[f1_df['url_host_name'] == host]['f1']
            if len(hf1) > 0:
                ax.hist(hf1, bins=20, color='steelblue', edgecolor='white', linewidth=0.3)
                ax.axvline(hf1.mean(), color='orange', linestyle='--', linewidth=1.5,
                           label=f"mean={hf1.mean():.2f}")
                ax.legend(fontsize=7)
            else:
                ax.text(0.5, 0.5, "no F1 data", ha='center', va='center',
                        transform=ax.transAxes, fontsize=9)
        else:
            ax.text(0.5, 0.5, "F1 not\ncomputed", ha='center', va='center',
                    transform=ax.transAxes, fontsize=9)

        ax.set_title(label, fontsize=8)
        ax.set_xlabel("Token F1", fontsize=8)

    plt.suptitle("F1 Distribution — Top 5 Hosts by Saved LLM Calls", fontsize=11, y=1.04)
    plt.tight_layout()
    plt.show()

## 6. Cluster Size Distribution

How are pages distributed across cluster sizes?  
Larger clusters = more LLM calls saved per representative.

In [ ]:
if manifest is not None:
    named_m = manifest[manifest['dripper_layout_id'].str.startswith('layout-', na=False)]
    failed_m = manifest[~manifest['dripper_layout_id'].str.startswith('layout-', na=False)]
    vc = named_m['dripper_layout_id'].value_counts()

    singletons  = (vc == 1).sum()
    multi       = (vc > 1).sum()
    mega        = (vc >= 1000).sum()  # clusters >= 1000 pages
    max_cluster = vc.iloc[0] if len(vc) > 0 else 0
    max_cluster_id = vc.index[0] if len(vc) > 0 else 'N/A'
    max_cluster_host = named_m[named_m['dripper_layout_id'] == max_cluster_id]['url_host_name'].iloc[0] \
                       if len(vc) > 0 else 'N/A'

    print(f"Cluster size statistics:")
    print(f"  Total clusters:         {len(vc):,}")
    print(f"  Singleton clusters:     {singletons:,}  ({singletons/len(vc)*100:.1f}%)")
    print(f"  Multi-page clusters:    {multi:,}  ({multi/len(vc)*100:.1f}%)")
    print(f"  Mega clusters (≥1000):  {mega}")
    print(f"  Largest cluster:        {max_cluster:,} pages  ({max_cluster_id})")
    print(f"  Largest cluster host:   {max_cluster_host}")
    print(f"  Non-clustered pages:    {len(failed_m):,}")

    # Histogram
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Panel 1: # clusters by size (log scale)
    ax = axes[0]
    ax.hist(vc.values, bins=np.logspace(0, np.log10(max(vc.values) + 1), 50),
            color='steelblue', edgecolor='white', linewidth=0.3)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Cluster size (pages)")
    ax.set_ylabel("# clusters")
    ax.set_title(f"Cluster Size Distribution ({len(vc):,} clusters)")
    # Annotate singleton vs multi
    ax.axvline(1.5, color='orange', linestyle='--', linewidth=1.5,
               label=f"Singletons: {singletons:,}")
    ax.legend(fontsize=9)

    # Panel 2: pages by cluster-size bucket
    bins_edges = [1, 2, 5, 10, 25, 50, 100, 250, 500, 1000, int(max(vc.values)) + 1]
    bin_labels = []
    page_counts = []
    for i in range(len(bins_edges) - 1):
        lo, hi = bins_edges[i], bins_edges[i+1]
        in_bucket = vc[(vc >= lo) & (vc < hi)]
        bin_labels.append(f"{lo}–{hi-1}" if hi - lo > 1 else str(lo))
        page_counts.append(int(in_bucket.sum()))

    ax = axes[1]
    bar_colors = ['#d9534f' if bins_edges[i] == 1 else
                  ('#e67e22' if bins_edges[i] < 10 else '#5cb85c')
                  for i in range(len(bin_labels))]
    bars = ax.bar(range(len(bin_labels)), page_counts, color=bar_colors,
                  edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(bin_labels)))
    ax.set_xticklabels(bin_labels, rotation=30, ha='right', fontsize=8)
    ax.set_xlabel("Cluster size bucket")
    ax.set_ylabel("Total pages in bucket")
    ax.set_title("Pages by Cluster Size Bucket")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f}K" if x >= 1000 else str(int(x))))
    for bar, v in zip(bars, page_counts):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, v * 1.01,
                    f"{v:,}", ha='center', va='bottom', fontsize=7)

    # Annotate the mega-cluster if it exists
    if max_cluster >= 1000:
        ax.annotate(
            f"Mega-cluster:\n{max_cluster:,} pages\n({max_cluster_host[:25]})",
            xy=(len(bin_labels) - 1, page_counts[-1]),
            xytext=(len(bin_labels) - 3, max(page_counts) * 0.7),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=8, color='red'
        )

    plt.suptitle("Cluster Size Analysis", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Manifest not loaded — skipping cluster size distribution.")

## 7. Example Content Comparison

Side-by-side: URL, Run A extracted content, Run B extracted content, F1 score.  
One representative cluster from each F1 tier: high (≥0.98), medium (0.90–0.95), low (<0.90).

In [ ]:
def show_comparison(row, label, preview_chars=400):
    """Print a side-by-side content comparison for one row."""
    f1  = row.get('f1', float('nan'))
    url = row.get('url', 'N/A')
    ca  = str(row.get('content_a') or '').strip()
    cb  = str(row.get('content_b') or '').strip()
    host = row.get('url_host_name', '')
    lid  = row.get('dripper_layout_id', '')

    print(f"{'='*80}")
    print(f"{label}")
    print(f"  URL:        {url}")
    print(f"  Host:       {host}    Layout: {lid}")
    print(f"  Token F1:   {f1:.4f}")
    print()
    print(f"  Run A (clustering):")
    print(f"    {repr(ca[:preview_chars])}")
    print()
    print(f"  Run B (standalone / ground truth):")
    print(f"    {repr(cb[:preview_chars])}")
    print()

if f1_df is not None and len(f1_df) > 0:
    # Pick one example from each tier
    tiers = [
        ("HIGH F1 (>= 0.98)",   f1_df[f1_df['f1'] >= 0.98]),
        ("MEDIUM F1 (0.90–0.95)", f1_df[(f1_df['f1'] >= 0.90) & (f1_df['f1'] < 0.95)]),
        ("LOW F1 (< 0.90)",     f1_df[f1_df['f1'] < 0.90]),
    ]

    shown = 0
    for label, subset in tiers:
        if len(subset) == 0:
            print(f"No examples for tier: {label}")
            continue
        # Pick the median example for robustness
        idx = subset['f1'].sub(subset['f1'].median()).abs().idxmin()
        show_comparison(subset.loc[idx], label)
        shown += 1
        if shown >= 3:
            break
else:
    print("F1 data not available — skipping content comparison.")
    print("Complete Sections 1 & 4 first.")

## 8. Summary Scorecard

In [ ]:
# Collect all scorecard numbers
sc_call_reduction = f"{call_reduction:.1f}%" if call_reduction > 0 else "N/A (jobs pending)"
sc_token_reduction = f"{token_reduction:.1f}%" if token_reduction > 0 else "N/A"
sc_mean_f1   = f"{f1_df['f1'].mean():.4f}" if f1_df is not None else "N/A"
sc_pct_95    = f"{(f1_df['f1'] >= 0.95).mean()*100:.1f}%" if f1_df is not None else "N/A"
sc_h100_a    = f"{h100h_a:,.0f}" if h100h_a > 0 else "N/A"
sc_h100_b    = f"{h100h_b:,.0f}" if h100h_b > 0 else "N/A"
sc_h100_save = f"{(h100h_b - h100h_a):,.0f}" if (h100h_a > 0 and h100h_b > 0) else "N/A"
sc_tput_a    = f"{throughput_a:.1f} pages/s" if throughput_a > 0 else "N/A"
sc_tput_b    = f"{throughput_b:.1f} pages/s" if throughput_b > 0 else "N/A"

scorecard = [
    ("LLM call reduction",       sc_call_reduction,  "← % of pages that skipped LLM via template"),
    ("Token reduction",          sc_token_reduction, "← total prompt+completion tokens saved"),
    ("Mean propagation F1",      sc_mean_f1,         "← vs Run B (standalone) as ground truth"),
    ("% pages with F1 >= 0.95",  sc_pct_95,          "← quality threshold"),
    ("Throughput Run A",         sc_tput_a,          "← pages/s with clustering"),
    ("Throughput Run B",         sc_tput_b,          "← pages/s standalone"),
    ("H100-hours Run A (proj.)", sc_h100_a,          "← full CC snapshot (~2.4B pages)"),
    ("H100-hours Run B (proj.)", sc_h100_b,          "← full CC snapshot (~2.4B pages)"),
    ("H100-hours saved",         sc_h100_save,       "← Run B − Run A"),
]

print()
print("╔" + "═"*72 + "╗")
print("║{:^72}║".format("SUMMARY SCORECARD — Clustering vs Standalone"))
print("╠" + "═"*72 + "╣")
for metric, value, note in scorecard:
    print(f"║  {metric:<35s}  {value:<12s}  {note:<18s}║")
print("╚" + "═"*72 + "╝")
print()
print("Dataset: chunk_0 / host_bucket=0000  |  44K pages  |  1,424 layout IDs")

In [ ]:
# Big-number visual scorecard
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

big_numbers = [
    ("Call\nReduction",    sc_call_reduction,  "#5cb85c"),
    ("Mean\nF1",           sc_mean_f1,         "steelblue"),
    ("H100-hours\nRun A",  sc_h100_a,          "#5cb85c"),
    ("H100-hours\nRun B",  sc_h100_b,          "#d9534f"),
]

for ax, (label, value, color) in zip(axes, big_numbers):
    ax.set_facecolor('#f8f9fa')
    ax.text(0.5, 0.60, value, ha='center', va='center',
            fontsize=22, fontweight='bold', color=color,
            transform=ax.transAxes)
    ax.text(0.5, 0.20, label, ha='center', va='center',
            fontsize=11, color='#555555',
            transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor('#cccccc')

plt.suptitle("Summary Scorecard — Layout Clustering vs Standalone Dripper",
             fontsize=12, y=1.05)
plt.tight_layout()
plt.show()